In [9]:
import os
import math
import time

import psycopg2
import psycopg2.extras
import requests

# -----------------------
# CONFIG
# -----------------------
DEBUG = False

DB_URL = os.environ.get(
    "DATABASE_URL",
    "postgresql://neondb_owner:npg_X71kIZjKPMcL"
    "@ep-twilight-cloud-agxexq52.c-2.eu-central-1.aws.neon.tech/DSPRO?sslmode=require"
)

SEARCH_URL = "https://api3.geo.admin.ch/rest/services/api/SearchServer"
IDENTIFY_URL = "https://api3.geo.admin.ch/rest/services/api/MapServer/identify"
HEIGHT_URL = "https://api3.geo.admin.ch/rest/services/height"
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

HEADERS = {"User-Agent": "swisstopo-enrich-fast/1.0"}

IDENTIFY_LAYERS = "all:" + ",".join([
    "ch.are.erreichbarkeit-oev",
    "ch.bfe.solarenergie-eignung-daecher",
])

SELECT_LISTINGS = """
SELECT listing_id, address
FROM listing_details
WHERE address IS NOT NULL
ORDER BY listing_id;
"""

UPSERT_SQL = """
INSERT INTO swisstopo_details (
    listing_id,
    oev_score,
    pt_distance_m,
    solar_irr,
    elevation_m,
    population,
    dist_autobahn_m,
    dist_school_m,
    lv95_east,
    lv95_north
) VALUES (
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s
)
ON CONFLICT (listing_id) DO UPDATE SET
    oev_score       = EXCLUDED.oev_score,
    pt_distance_m   = EXCLUDED.pt_distance_m,
    solar_irr       = EXCLUDED.solar_irr,
    elevation_m     = EXCLUDED.elevation_m,
    population      = EXCLUDED.population,
    dist_autobahn_m = EXCLUDED.dist_autobahn_m,
    dist_school_m   = EXCLUDED.dist_school_m,
    lv95_east       = EXCLUDED.lv95_east,
    lv95_north      = EXCLUDED.lv95_north;
"""

# optional für Tests:
# SELECT_LISTINGS = """
# SELECT listing_id, address
# FROM listing_details
# WHERE address IS NOT NULL
# ORDER BY listing_id
# LIMIT 100;
# """


# -----------------------
# HELPERS
# -----------------------
def safe_num(x, default=None):
    try:
        if x is None:
            return default
        v = float(x)
        if math.isnan(v) or math.isinf(v):
            return default
        return v
    except Exception:
        return default


def safe_int(x, default=None):
    v = safe_num(x, None)
    if v is None:
        return default
    try:
        return int(round(v))
    except Exception:
        return default


def _request(url, *, params=None, data=None, method="GET", timeout=15):
    for attempt in range(2):
        try:
            if method == "POST":
                r = requests.post(url, data=data, timeout=timeout, headers=HEADERS)
            else:
                r = requests.get(url, params=params, timeout=timeout, headers=HEADERS)

            if r.status_code == 200:
                return r

            if attempt == 0 and DEBUG:
                tail = url.rstrip("/").split("/")[-1] or url
                print(f"    ⚠ HTTP {r.status_code} {tail}: {r.text[:160]}")
            return None

        except requests.exceptions.ConnectionError as e:
            if attempt == 0:
                if DEBUG:
                    print(f"    ⚠ connection error, retry in 2s: {e}")
                time.sleep(2)
        except Exception as e:
            if DEBUG:
                print(f"    ⚠ request error: {e}")
            return None

    return None


def _get_json(url, params, timeout=15):
    r = _request(url, params=params, method="GET", timeout=timeout)
    if not r:
        return None
    try:
        return r.json()
    except Exception:
        return None


def _post_json(url, data, timeout=20):
    r = _request(url, data=data, method="POST", timeout=timeout)
    if not r:
        return None
    try:
        return r.json()
    except Exception:
        return None


def lv95_to_wgs84(east, north):
    y = (east - 2_600_000) / 1_000_000
    x = (north - 1_200_000) / 1_000_000

    lon_a = (
        2.6779094
        + 4.728982 * y
        + 0.791484 * y * x
        + 0.1306 * y * x * x
        - 0.0436 * y**3
    )
    lat_a = (
        16.9023892
        + 3.238272 * x
        - 0.270978 * y**2
        - 0.002528 * x**2
        - 0.0447 * y**2 * x
        - 0.014 * x**3
    )

    return lat_a * 100 / 36, lon_a * 100 / 36


# -----------------------
# GEOCODING / HEIGHT / IDENTIFY
# -----------------------
def geocode(address):
    d = _get_json(SEARCH_URL, {
        "searchText": address,
        "type": "locations",
        "sr": 2056,
        "limit": 1,
    })
    if not d or not d.get("results"):
        return None

    attrs = d["results"][0].get("attrs", {})

    # SearchServer liefert hier in deinem Fall:
    # y = east, x = north
    east = safe_num(attrs.get("y"))
    north = safe_num(attrs.get("x"))

    if east is None or north is None:
        return None

    return east, north


def get_elevation(east, north):
    d = _get_json(HEIGHT_URL, {"easting": east, "northing": north}, timeout=15)
    if not d:
        return None
    return safe_num(d.get("height"))


def identify(east, north):
    return _get_json(IDENTIFY_URL, {
        "geometry": f"{east},{north}",
        "geometryType": "esriGeometryPoint",
        "layers": IDENTIFY_LAYERS,
        "tolerance": 1000,
        "returnGeometry": "false",
        "sr": 2056,
        "imageDisplay": "1000,1000,96",
        "mapExtent": f"{east-1000},{north-1000},{east+1000},{north+1000}",
    }, timeout=20) or {"results": []}


def parse_identify(results):
    out = {
        "oev_score": None,
        "solar_irr": None,
    }

    for item in results:
        layer = item.get("layerBodId", "")
        attr = item.get("attributes", {}) or {}

        if layer == "ch.are.erreichbarkeit-oev":
            val = safe_num(attr.get("oev_erreichb_ewap"))
            if val is not None and (out["oev_score"] is None or val > out["oev_score"]):
                out["oev_score"] = val

        elif layer == "ch.bfe.solarenergie-eignung-daecher":
            irr = safe_num(attr.get("gstrahlung"))
            if irr is not None and (out["solar_irr"] is None or irr > out["solar_irr"]):
                out["solar_irr"] = irr

    return out


# -----------------------
# FAST DISTANCE HELPERS (Overpass)
# -----------------------
def get_dist_pt(east, north):
    lat, lon = lv95_to_wgs84(east, north)

    def _query(radius):
        q = f"""
        [out:json][timeout:20];
        (
          node["highway"="bus_stop"](around:{radius},{lat},{lon});
          node["public_transport"~"^(platform|stop_position|station)$"](around:{radius},{lat},{lon});
          node["railway"~"^(station|halt|tram_stop)$"](around:{radius},{lat},{lon});
          way["public_transport"~"^(platform|station)$"](around:{radius},{lat},{lon});
          way["railway"~"^(station|halt|tram_stop)$"](around:{radius},{lat},{lon});
          relation["public_transport"="station"](around:{radius},{lat},{lon});
        );
        out center;
        """
        res = _post_json(OVERPASS_URL, {"data": q}, timeout=25)
        if not res or not res.get("elements"):
            return None

        best = float("inf")
        cos_lat = math.cos(math.radians(lat))

        for el in res["elements"]:
            slat = el.get("lat") or (el.get("center") or {}).get("lat")
            slon = el.get("lon") or (el.get("center") or {}).get("lon")
            if slat is None or slon is None:
                continue

            dlat = (slat - lat) * 111_000
            dlon = (slon - lon) * 111_000 * cos_lat
            best = min(best, math.hypot(dlat, dlon))

        return round(best, 1) if best != float("inf") else None

    for radius in (1200, 3000, 8000):
        d = _query(radius)
        if d is not None:
            return d

    return None


def get_dist_autobahn(east, north):
    lat, lon = lv95_to_wgs84(east, north)
    cos_lat = math.cos(math.radians(lat))

    for radius in (20000, 60000):
        q = f"""
        [out:json][timeout:20];
        node["highway"="motorway_junction"](around:{radius},{lat},{lon});
        out;
        """
        res = _post_json(OVERPASS_URL, {"data": q}, timeout=25)
        if not res or not res.get("elements"):
            continue

        best = float("inf")
        for el in res["elements"]:
            slat = el.get("lat")
            slon = el.get("lon")
            if slat is None or slon is None:
                continue

            dlat = (slat - lat) * 111_000
            dlon = (slon - lon) * 111_000 * cos_lat
            best = min(best, math.hypot(dlat, dlon))

        if best != float("inf"):
            return round(best, 1)

    return None


def get_dist_school(east, north):
    lat, lon = lv95_to_wgs84(east, north)
    cos_lat = math.cos(math.radians(lat))

    def _query(radius):
        q = f"""
        [out:json][timeout:20];
        (
          node["amenity"~"^(school|kindergarten)$"](around:{radius},{lat},{lon});
          way["amenity"~"^(school|kindergarten)$"](around:{radius},{lat},{lon});
        );
        out center;
        """
        res = _post_json(OVERPASS_URL, {"data": q}, timeout=25)
        if not res or not res.get("elements"):
            return None

        best = float("inf")
        for el in res["elements"]:
            slat = el.get("lat") or (el.get("center") or {}).get("lat")
            slon = el.get("lon") or (el.get("center") or {}).get("lon")
            if slat is None or slon is None:
                continue

            dlat = (slat - lat) * 111_000
            dlon = (slon - lon) * 111_000 * cos_lat
            best = min(best, math.hypot(dlat, dlon))

        return round(best, 1) if best != float("inf") else None

    return _query(3000) or _query(9000)


# -----------------------
# POPULATION
# -----------------------
def get_population(east, north):
    """
    Schnellere Punktabfrage nur für Bevölkerungs-Layer.
    Kein WMS.
    """
    data = _get_json(IDENTIFY_URL, {
        "geometry": f"{east},{north}",
        "geometryType": "esriGeometryPoint",
        "layers": "all:ch.bfs.volkszaehlung-bevoelkerungsstatistik_einwohner",
        "tolerance": 10,
        "returnGeometry": "false",
        "sr": 2056,
        "imageDisplay": "100,100,96",
        "mapExtent": f"{east-10},{north-10},{east+10},{north+10}",
    }, timeout=15) or {"results": []}

    pop_total = 0
    found = False

    for item in data.get("results", []):
        attr = item.get("attributes", {}) or {}
        number_val = safe_int(attr.get("number"), None)
        year_val = attr.get("i_year")

        if number_val is None:
            continue

        if year_val is None or year_val == 2024:
            pop_total += number_val
            found = True

    return pop_total if found else None


# -----------------------
# MAIN
# -----------------------
def main():
    conn = psycopg2.connect(DB_URL)
    conn.autocommit = False

    cur = conn.cursor()
    dcur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    dcur.execute(SELECT_LISTINGS)
    rows = dcur.fetchall()
    total = len(rows)

    print(f"Enriching {total} listings...")

    ok = 0
    skip = 0
    fail = 0

    try:
        for i, row in enumerate(rows, start=1):
            listing_id = row["listing_id"]
            address = row["address"]

            try:
                print(f"[{i}/{total}] id={listing_id}")

                coords = geocode(address)
                if not coords:
                    print("   -> geocode failed, skip")
                    skip += 1
                    continue

                east, north = coords

                ident = identify(east, north)
                parsed = parse_identify(ident.get("results") or [])

                elevation_m = get_elevation(east, north)
                population = get_population(east, north)
                pt_distance_m = get_dist_pt(east, north)
                dist_autobahn_m = get_dist_autobahn(east, north)
                dist_school_m = get_dist_school(east, north)

                cur.execute(
                    UPSERT_SQL,
                    (
                        listing_id,
                        parsed["oev_score"],
                        pt_distance_m,
                        parsed["solar_irr"],
                        elevation_m,
                        population,
                        dist_autobahn_m,
                        dist_school_m,
                        east,
                        north,
                    )
                )
                conn.commit()
                ok += 1

                print(
                    f"   ✓ oev={parsed['oev_score']} "
                    f"pt={pt_distance_m}m "
                    f"solar={parsed['solar_irr']} "
                    f"elev={elevation_m} "
                    f"pop={population} "
                    f"auto={dist_autobahn_m} "
                    f"school={dist_school_m}"
                )

                time.sleep(0.15)

            except Exception as e:
                conn.rollback()
                fail += 1
                print(f"   ✗ error on id={listing_id}: {e}")

    finally:
        cur.close()
        dcur.close()
        conn.close()

    print(f"Done. ok={ok} skipped={skip} errors={fail}")


if __name__ == "__main__":
    main()

Enriching 9839 listings...
[1/9839] id=1
   ✓ oev=39813.0 pt=120.7m solar=755363.0 elev=572.9 pop=93 auto=2795.5 school=367.3
[2/9839] id=2
   ✓ oev=15535.0 pt=57.6m solar=591051.0 elev=672.3 pop=107 auto=None school=465.5
[3/9839] id=3
   ✓ oev=50158.0 pt=98.5m solar=1691044.0 elev=417.9 pop=252 auto=738.9 school=265.0
[4/9839] id=4
   ✓ oev=12617.0 pt=Nonem solar=2690372.0 elev=439.6 pop=236 auto=2006.9 school=34.3
[5/9839] id=5
   ✓ oev=36690.0 pt=98.7m solar=3454607.0 elev=430.4 pop=129 auto=485.5 school=276.2
[6/9839] id=6
   ✓ oev=1437.0 pt=279.1m solar=1406020.0 elev=743.6 pop=48 auto=945.8 school=202.4
[7/9839] id=7
   ✓ oev=55264.0 pt=72.9m solar=1127534.0 elev=486.9 pop=92 auto=651.3 school=None
[8/9839] id=8
   ✓ oev=58387.0 pt=145.3m solar=4801235.0 elev=546.0 pop=208 auto=676.1 school=103.2
[9/9839] id=9
   ✓ oev=55264.0 pt=92.9m solar=1066025.0 elev=459.4 pop=125 auto=508.7 school=None
[10/9839] id=10
   ✓ oev=263.0 pt=854.9m solar=487851.0 elev=796.5 pop=14 auto=None sch

KeyboardInterrupt: 

In [2]:
import os
import math
import time

import psycopg2
import psycopg2.extras
import requests

# -----------------------
# CONFIG
# -----------------------
DEBUG = False

DB_URL = os.environ.get(
    "DATABASE_URL",
    "postgresql://neondb_owner:npg_X71kIZjKPMcL"
    "@ep-twilight-cloud-agxexq52.c-2.eu-central-1.aws.neon.tech/DSPRO?sslmode=require"
)

SEARCH_URL = "https://api3.geo.admin.ch/rest/services/api/SearchServer"
IDENTIFY_URL = "https://api3.geo.admin.ch/rest/services/api/MapServer/identify"
HEIGHT_URL = "https://api3.geo.admin.ch/rest/services/height"

HEADERS = {"User-Agent": "swisstopo-enrich-fast/1.0"}

IDENTIFY_LAYERS = "all:" + ",".join([
    "ch.are.erreichbarkeit-oev",
    "ch.bfe.solarenergie-eignung-daecher",
])

SELECT_LISTINGS = """
SELECT listing_id, address
FROM listing_details
WHERE address IS NOT NULL
ORDER BY listing_id;
"""

UPSERT_SQL = """
INSERT INTO swisstopo_details (
    listing_id,
    oev_score,
    pt_distance_m,
    solar_irr,
    elevation_m,
    population,
    dist_autobahn_m,
    dist_school_m,
    lv95_east,
    lv95_north
) VALUES (
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s
)
ON CONFLICT (listing_id) DO UPDATE SET
    oev_score       = EXCLUDED.oev_score,
    pt_distance_m   = EXCLUDED.pt_distance_m,
    solar_irr       = EXCLUDED.solar_irr,
    elevation_m     = EXCLUDED.elevation_m,
    population      = EXCLUDED.population,
    dist_autobahn_m = EXCLUDED.dist_autobahn_m,
    dist_school_m   = EXCLUDED.dist_school_m,
    lv95_east       = EXCLUDED.lv95_east,
    lv95_north      = EXCLUDED.lv95_north;
"""

# optional für Tests:
# SELECT_LISTINGS = """
# SELECT listing_id, address
# FROM listing_details
# WHERE address IS NOT NULL
# ORDER BY listing_id
# LIMIT 100;
# """


# -----------------------
# HELPERS
# -----------------------
def safe_num(x, default=None):
    try:
        if x is None:
            return default
        v = float(x)
        if math.isnan(v) or math.isinf(v):
            return default
        return v
    except Exception:
        return default


def safe_int(x, default=None):
    v = safe_num(x, None)
    if v is None:
        return default
    try:
        return int(round(v))
    except Exception:
        return default


def _request(url, *, params=None, timeout=15):
    for attempt in range(2):
        try:
            r = requests.get(url, params=params, timeout=timeout, headers=HEADERS)

            if r.status_code == 200:
                return r

            if attempt == 0 and DEBUG:
                tail = url.rstrip("/").split("/")[-1] or url
                print(f"    ⚠ HTTP {r.status_code} {tail}: {r.text[:160]}")
            return None

        except requests.exceptions.ConnectionError as e:
            if attempt == 0:
                if DEBUG:
                    print(f"    ⚠ connection error, retry in 2s: {e}")
                time.sleep(2)
        except Exception as e:
            if DEBUG:
                print(f"    ⚠ request error: {e}")
            return None

    return None


def _get_json(url, params, timeout=15):
    r = _request(url, params=params, timeout=timeout)
    if not r:
        return None
    try:
        return r.json()
    except Exception:
        return None


# -----------------------
# GEOCODING / HEIGHT / IDENTIFY
# -----------------------
def geocode(address):
    d = _get_json(SEARCH_URL, {
        "searchText": address,
        "type": "locations",
        "sr": 2056,
        "limit": 1,
    }, timeout=15)

    if not d or not d.get("results"):
        return None

    attrs = d["results"][0].get("attrs", {})

    # In deinem Fall liefert SearchServer:
    # y = east, x = north
    east = safe_num(attrs.get("y"))
    north = safe_num(attrs.get("x"))

    if east is None or north is None:
        return None

    return east, north


def get_elevation(east, north):
    d = _get_json(
        HEIGHT_URL,
        {"easting": east, "northing": north},
        timeout=15
    )
    if not d:
        return None
    return safe_num(d.get("height"))


def identify(east, north):
    return _get_json(
        IDENTIFY_URL,
        {
            "geometry": f"{east},{north}",
            "geometryType": "esriGeometryPoint",
            "layers": IDENTIFY_LAYERS,
            "tolerance": 1000,
            "returnGeometry": "false",
            "sr": 2056,
            "imageDisplay": "1000,1000,96",
            "mapExtent": f"{east-1000},{north-1000},{east+1000},{north+1000}",
        },
        timeout=20
    ) or {"results": []}


def identify_population(east, north):
    return _get_json(
        IDENTIFY_URL,
        {
            "geometry": f"{east},{north}",
            "geometryType": "esriGeometryPoint",
            "layers": "all:ch.bfs.volkszaehlung-bevoelkerungsstatistik_einwohner",
            "tolerance": 10,
            "returnGeometry": "false",
            "sr": 2056,
            "imageDisplay": "100,100,96",
            "mapExtent": f"{east-10},{north-10},{east+10},{north+10}",
        },
        timeout=15
    ) or {"results": []}


def parse_identify(results):
    out = {
        "oev_score": None,
        "solar_irr": None,
    }

    for item in results:
        layer = item.get("layerBodId", "")
        attr = item.get("attributes", {}) or {}

        if layer == "ch.are.erreichbarkeit-oev":
            val = safe_num(attr.get("oev_erreichb_ewap"))
            if val is not None and (out["oev_score"] is None or val > out["oev_score"]):
                out["oev_score"] = val

        elif layer == "ch.bfe.solarenergie-eignung-daecher":
            irr = safe_num(attr.get("gstrahlung"))
            if irr is not None and (out["solar_irr"] is None or irr > out["solar_irr"]):
                out["solar_irr"] = irr

    return out


def parse_population(results):
    pop_total = 0
    found = False

    for item in results:
        attr = item.get("attributes", {}) or {}
        number_val = safe_int(attr.get("number"), None)
        year_val = attr.get("i_year")

        if number_val is None:
            continue

        if year_val is None or year_val == 2024:
            pop_total += number_val
            found = True

    return pop_total if found else None


# -----------------------
# MAIN
# -----------------------
def main():
    conn = psycopg2.connect(DB_URL)
    conn.autocommit = False

    cur = conn.cursor()
    dcur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    dcur.execute(SELECT_LISTINGS)
    rows = dcur.fetchall()
    total = len(rows)

    print(f"Enriching {total} listings...")

    ok = 0
    skip = 0
    fail = 0

    try:
        for i, row in enumerate(rows, start=1):
            listing_id = row["listing_id"]
            address = row["address"]

            try:
                print(f"[{i}/{total}] id={listing_id}")

                coords = geocode(address)
                if not coords:
                    print("   -> geocode failed, skip")
                    skip += 1
                    continue

                east, north = coords

                ident = identify(east, north)
                parsed = parse_identify(ident.get("results") or [])

                pop_ident = identify_population(east, north)
                population = parse_population(pop_ident.get("results") or [])

                elevation_m = get_elevation(east, north)

                pt_distance_m = None
                dist_autobahn_m = None
                dist_school_m = None

                cur.execute(
                    UPSERT_SQL,
                    (
                        listing_id,
                        parsed["oev_score"],
                        pt_distance_m,
                        parsed["solar_irr"],
                        elevation_m,
                        population,
                        dist_autobahn_m,
                        dist_school_m,
                        east,
                        north,
                    )
                )
                conn.commit()
                ok += 1

                print(
                    f"   ✓ oev={parsed['oev_score']} "
                    f"pt=None "
                    f"solar={parsed['solar_irr']} "
                    f"elev={elevation_m} "
                    f"pop={population} "
                    f"auto=None "
                    f"school=None"
                )

                time.sleep(0.05)

            except Exception as e:
                conn.rollback()
                fail += 1
                print(f"   ✗ error on id={listing_id}: {e}")

    finally:
        cur.close()
        dcur.close()
        conn.close()

    print(f"Done. ok={ok} skipped={skip} errors={fail}")


if __name__ == "__main__":
    main()

Enriching 9839 listings...
[1/9839] id=1
   ✓ oev=39813.0 pt=None solar=504349.0 elev=572.9 pop=93 auto=None school=None
[2/9839] id=2
   ✓ oev=15535.0 pt=None solar=591051.0 elev=672.3 pop=107 auto=None school=None
[3/9839] id=3
   ✓ oev=50158.0 pt=None solar=888893.0 elev=417.9 pop=252 auto=None school=None
[4/9839] id=4
   ✓ oev=12617.0 pt=None solar=2530701.0 elev=439.6 pop=236 auto=None school=None
[5/9839] id=5
   ✓ oev=36690.0 pt=None solar=4526536.0 elev=430.4 pop=129 auto=None school=None
[6/9839] id=6
   ✓ oev=1437.0 pt=None solar=1406020.0 elev=743.6 pop=48 auto=None school=None
[7/9839] id=7
   ✓ oev=55264.0 pt=None solar=1127534.0 elev=486.9 pop=92 auto=None school=None
[8/9839] id=8
   ✓ oev=58387.0 pt=None solar=4801235.0 elev=546.0 pop=208 auto=None school=None
[9/9839] id=9
   ✓ oev=55264.0 pt=None solar=1039681.0 elev=459.4 pop=125 auto=None school=None
[10/9839] id=10
   ✓ oev=263.0 pt=None solar=487851.0 elev=796.5 pop=14 auto=None school=None
[11/9839] id=11
   ✓ o